# 11주차 · 시계열 예측

**⏱ 60분** · 미국 월별 항공 여객 1990년 1월 ~ 2004년 4월

## 🎯 이번 주의 질문

2001년 9월 미국의 항공 여객이 갑자기 크게 줄었습니다. 항공사와 정부가 감편·지원 규모를 정하려면 얼마나 줄었는지를 숫자로 알아야 합니다. 그런데 2001년 9월 전과 후를 그대로 비교하면 안 됩니다. 항공 여객은 해마다 늘고 있었고, 달마다 크게 오르내리기 때문입니다. 그래서 미국 교통통계국(BTS)은 2001년 8월까지의 기록으로 **이전 추세가 이어졌을 때의 값(기준선)을 예측**하고, 실제 값과의 차이를 그 감소의 크기로 보았습니다. 이 기준선을 만드는 일이 시간 순서로 쌓인 과거 기록으로 앞으로의 값을 내는 **시계열 예측(time series forecasting)** 입니다.

> **2001년 9월 이후 미국 항공 여객은 이전 추세로 예측한 기준선보다 얼마나 적었고, 그 차이는 얼마나 이어졌나?**

**이번 주에 할 일**
1. 자료의 열을 읽고 월별 시계열로 만든다 (1절)
2. 2001년 8월까지와 9월 이후로 나누고, 2001년 8월까지의 마지막 12개월을 테스트 기간으로 떼어 둔다 (2절)
3. 방법마다 테스트 기간 12개월을 예측하고 그림으로 본다 (3~5절)
   - naive 예측, seasonal naive 예측 (3절)
   - 자기회귀 AR(4), 계절 자기회귀 (4절)
   - 선형 추세 모형, 선형 추세 + 계절성 모형 (5절)
4. 테스트 기간의 오차 지표로 방법을 비교해 하나를 고른다 (6절)
5. 고른 방법을 2001년 8월까지 전체로 다시 맞춰 기준선을 예측하고 실제와 비교한다 (7·8절)

**데이터:** BTS 보고서 *Estimated Impacts of September 11th on US Travel*의 월별 교통량(Shmueli 외 교과서 연습문제 자료). 교재는 항공, 랩은 철도를 씁니다.

**반복할 패턴:** 방법마다 맞추기(`fit`) → 테스트 기간 예측(`predict`) → 그림 → 오차 지표로 비교 → 기준선과 실제의 차이

배경이론: [11주차 이론](https://balab-pknu.github.io/bizanalytics/theory/week11.html)

| 셀 표시 | 할 일 |
|---|---|
| `# ✍️ 직접 입력` | 위에 보이는 코드를 직접 쳐서 실행합니다 |
| `# ▶ 그대로 실행하세요` | 준비·그림 코드입니다. 실행만 합니다 |
| `# 📝` | 과제입니다. 힌트를 보고 직접 씁니다 |

> **실습본입니다.** 먼저 메뉴 **파일 → Drive에 사본 저장**을 누르세요. 사본을 저장하지 않으면 입력한 코드가 사라집니다.
> 맨 위 준비 셀부터 순서대로 `Shift + Enter`로 실행합니다. ▶ 셀은 그대로 실행하고, ✍️ 셀은 위에 보여 준 코드를
> 빈 셀에 손으로 타이핑하고, 빈칸(`___`)은 채워서 실행합니다. 📝 과제는 힌트를 보고 스스로 풉니다.

In [ ]:
# ▶ 준비 셀 — 그대로 실행하세요. 미국 월별 교통량을 읽습니다.
import numpy as np
import pandas as pd
pd.set_option("display.max_rows", 500)       # 표를 줄임표(...) 없이 끝까지 보여 줌
pd.set_option("display.max_columns", 50)
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

import os
import sys
for _p in (".", "..", "../.."):
    if os.path.exists(os.path.join(_p, "balab.py")):
        sys.path.insert(0, _p); break
else:
    !wget -q https://raw.githubusercontent.com/BALAB-PKNU/bizanalytics/main/balab.py

from balab import load
df = load("sept11_travel")
print("준비 완료!", df.shape)

✅ `준비 완료! (172, 4)`가 나오면 됩니다.

---
## 1. 자료의 열과 월별 시계열

시계열은 일정한 간격으로 기록한 값을 시간 순서대로 놓은 것입니다. 표의 앞부분부터 봅니다.

```python
df.head()
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 한 행이 한 달입니다. 열의 뜻은 다음과 같습니다.

| 열 | 뜻 | 단위 |
|---|---|---|
| `Month` | 기록한 달. `Jan-90`은 1990년 1월 | 월 이름-연도 두 자리 글자 |
| `Air RPM (000s)` | 항공 유상 여객마일(revenue passenger miles). 돈을 낸 승객 1명을 1마일 태우면 1입니다. 승객 수와 이동 거리를 함께 담은 항공 수요의 표준 지표입니다 | 천 여객마일 |
| `Rail PM` | 철도 여객마일(passenger miles). 승객 1명을 1마일 태우면 1입니다 | 여객마일 |
| `VMT (billions)` | 자동차 주행거리(vehicle miles traveled). 차 1대가 1마일 달리면 1입니다 | 십억 마일 |

교재는 `Air RPM (000s)`, 랩은 `Rail PM`을 씁니다.

`Month`는 글자라 날짜 계산이나 달 꺼내기를 할 수 없으므로 날짜로 바꿉니다. `pd.to_datetime`에 글자의 형식(`%b-%y` = 월 이름-연도 두 자리)을 알려 줍니다.

```python
df["Month"] = pd.to_datetime(df["Month"], format="%b-%y")
df = df.set_index("Month")
print(df.index.min(), df.index.max(), len(df))
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ `1990-01-01 00:00:00 2004-04-01 00:00:00 172`가 나오면 맞습니다. 날짜를 행 이름(index)으로 두었고, 빠진 달 없이 172개월입니다.

`Air RPM (000s)`는 천 단위라 숫자가 크므로, 100만으로 나눠 **십억 여객마일** 단위의 열 `air`를 만듭니다.

```python
df["air"] = df["Air RPM (000s)"] / 1e6
df["air"].head(3).round(2)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ 1990년 1월이 **35.15**(십억 여객마일)이면 맞습니다.

In [ ]:
# ▶ 그대로 실행하세요 — 월별 항공 여객, 빨간 점선이 2001년 9월
plt.figure(figsize=(9, 3.5))
plt.plot(df.index, df["air"])
plt.axvline(pd.Timestamp("2001-09-01"), color="red", linestyle="--")
plt.xlabel("Month")
plt.ylabel("Air RPM (billions)")
plt.show()

💬 **결과 해설**
- 항공 여객은 1990년대 내내 꾸준히 늘었습니다. 긴 기간에 걸친 이 흐름이 **추세**입니다.
- 해마다 같은 모양의 오르내림이 되풀이됩니다. 일정한 주기로 되풀이되는 패턴이 **계절성**입니다.
- 2001년 8월 69.0에서 9월 39.1로 떨어졌습니다. 그런데 9월은 원래 8월보다 적은 달이고, 해마다 여객이 늘던 중이었습니다. 이 낙폭이 평소의 계절 변동을 넘는 감소인지는 그래프만으로 알 수 없습니다. "이전 추세가 이어졌다면 9월은 얼마였을까"가 필요합니다.

---
## 2. 기간 나누기

기준선은 2001년 9월 이후의 변화가 들어 있지 않은 기록, 곧 **2001년 8월까지의 140개월**로만 만들 수 있습니다. 2001년 8월까지를 `pre`, 2001년 9월부터를 `post`로 나눕니다. `df.loc[:"2001-08"]`은 2001년 8월까지, `df.loc["2001-09":]`는 2001년 9월부터의 행입니다.

```python
pre = df.loc[:"2001-08"]
post = df.loc["2001-09":]
print(len(pre), len(post))
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ `140 32`가 나오면 맞습니다.

기준선을 만들 방법은 여러 가지입니다. 어느 방법이 앞을 잘 맞히는지 보려면 2001년 8월까지의 기록에서도 앞을 맞혀 봐야 합니다. 그 140개월의 **마지막 12개월**(2000년 9월 ~ 2001년 8월)을 테스트 기간 `test`로 떼어 두고, 방법은 그 앞 128개월인 학습 기간 `train`으로만 맞춥니다. 질문의 중심이 2001년 9월 이후 첫 1년이므로 1년 앞을 맞히는 능력으로 고릅니다. 미래는 과거 뒤에 오므로 시간 순서대로 자릅니다. 테스트 기간의 실제 값은 `y_test`로 둡니다.

```python
train = pre.iloc[:-12]
test = pre.iloc[-12:]
y_test = test["air"]
print(train.index.max(), test.index.min(), len(train), len(test))
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ 학습 기간이 2000년 8월까지 **128**개월, 테스트 기간이 2000년 9월부터 **12**개월이면 맞습니다.

계절성이 몇 달마다 되풀이되는지(계절 주기)를 정합니다. 학습 기간을 달(1~12월)로 묶어 평균을 봅니다. `train.index.month`는 각 행의 달입니다.

```python
train.groupby(train.index.month)["air"].mean().round(1)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ 2월 **39.4**, 8월 **56.3**이면 맞습니다.

💬 2월에 가장 적고 7·8월에 가장 많습니다. 9월은 8월보다 약 11 적습니다. 이 모양이 해마다 되풀이되므로 계절 주기는 **12개월**입니다.

---
## 3. naive 예측과 seasonal naive 예측

지금부터 방법마다 테스트 기간 12개월을 예측하고 그림으로 봅니다. 예측값은 방법 이름을 붙여 `preds`에, 모형이 학습 기간에서 계산한 **적합값**은 `fits`에 모읍니다. 그림 함수 `plot_forecast`는 실제 값(회색 점), 적합값(실선), 테스트 기간의 예측(점선)을 1997년부터 그립니다. 세로선이 테스트 기간의 시작입니다.

In [ ]:
# ▶ 그대로 실행하세요 — 예측값·적합값을 모을 곳과 그림 함수
preds = {}
fits = {}


def plot_forecast(name):
    plt.figure(figsize=(9, 3.5))
    plt.plot(pre.loc["1997":].index, pre.loc["1997":, "air"], "o", color="gray", markersize=3, label="Actual")
    if name in fits:
        plt.plot(fits[name].loc["1997":].index, fits[name].loc["1997":], color="C0", label="Fitted (training)")
    plt.plot(preds[name].index, preds[name], "--o", color="C0", markersize=4, label="Forecast (test)")
    plt.axvline(test.index[0], color="black", linewidth=0.8)
    plt.title(name)
    plt.ylabel("Air RPM (billions)")
    plt.legend()
    plt.show()

**naive 예측**은 학습 기간의 마지막 값(2000년 8월)을 앞으로 12개월 내내 씁니다. `pd.Series(값, index=test.index)`는 테스트 기간의 달마다 값을 넣은 표입니다.

```python
preds["naive"] = pd.Series(train["air"].iloc[-1], index=test.index)
preds["naive"].head(3).round(2)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ 2000년 9월의 예측이 **66.92**(2000년 8월의 값)이면 맞습니다.

In [ ]:
# ▶ 그대로 실행하세요 — naive 예측 그림
plot_forecast("naive")

💬 성수기인 8월의 값을 가을·겨울에도 그대로 써서, 9월 이후 실제 값보다 크게 위에 있습니다.

**seasonal naive 예측**은 한 주기(12개월) 전 같은 달의 값을 씁니다. 테스트 기간이 12개월이므로 학습 기간의 마지막 12개월을 그대로 씁니다. `tolist()`는 값을 리스트로 꺼냅니다.

```python
preds["seasonal naive"] = pd.Series(train["air"].iloc[-12:].tolist(), index=test.index)
preds["seasonal naive"].head(3).round(2)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ 2000년 9월의 예측이 **53.73**(1999년 9월의 값)이면 맞습니다.

In [ ]:
# ▶ 그대로 실행하세요 — seasonal naive 예측 그림
plot_forecast("seasonal naive")

💬 계절 모양은 따라가지만, 해마다 여객이 늘어나는 만큼은 반영하지 못해 테스트 기간에는 대체로 실제보다 조금 아래에 있습니다.

---
## 4. 자기회귀 모형

**자기회귀(autoregression)** 는 설명 변수가 같은 시계열의 직전 몇 기간의 값(lag)인 회귀 모형입니다. 먼저 표에 직전 값 열을 붙입니다. `shift(k)`는 k칸 앞, 곧 k개월 전의 값을 가져옵니다. AR(4)에 쓸 직전 4개월(`lag1`~`lag4`)과 계절 자기회귀에 쓸 1년 전 같은 달(`lag12`)의 열을 만듭니다.

```python
df["lag1"] = df["air"].shift(1)
df["lag2"] = df["air"].shift(2)
df["lag3"] = df["air"].shift(3)
df["lag4"] = df["air"].shift(4)
df["lag12"] = df["air"].shift(12)
df[["air", "lag1", "lag2", "lag3", "lag4", "lag12"]].head(6).round(2)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ 1990년 5월 행의 `lag4`가 1990년 1월의 값 **35.15**이면 맞습니다.

💬 앞쪽 행은 직전 값이 없어 비어 있습니다(`NaN`). `lag12`는 1991년 1월부터 채워집니다.

### 자기회귀 AR(4)

직전 4개월의 값을 입력, 그 달의 값을 출력으로 회귀를 맞춥니다. 열은 전체 표 `df`에 붙였으므로 학습 기간의 행은 `df.loc[train.index]`로 꺼냅니다. 처음 4개월은 입력이 모자라 `iloc[4:]`로 뺍니다.

```python
lags = ["lag1", "lag2", "lag3", "lag4"]
X_train = df.loc[train.index, lags].iloc[4:]
y_train = train["air"].iloc[4:]
X_train.shape
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ `(124, 4)`가 나오면 맞습니다. 학습 기간 128개월에서 처음 4개월을 뺀 124개월입니다.

```python
model = LinearRegression().fit(X_train, y_train)
fits["AR(4)"] = pd.Series(model.predict(X_train), index=X_train.index)
print(model.intercept_)
pd.Series(model.coef_, index=lags).round(3)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ 절편 약 **6.66**, `lag1` **0.706**, `lag2` 0.201이면 맞습니다.

💬 직전 달의 값에 가장 큰 계수가 붙었습니다.

테스트 기간의 예측은 한 달씩 이어서 냅니다. 2000년 10월을 예측하려면 2000년 9월의 값이 입력에 필요한데, 테스트 기간의 실제 값은 쓰지 않으므로 9월의 예측값을 대신 넣습니다. 이렇게 앞 달의 예측값을 다음 달의 입력으로 쓰는 것이 **다단계 예측**입니다. `history`에 학습 기간의 값을 담고, 한 달을 예측할 때마다 그 예측값을 `history` 뒤에 붙입니다. `history[-1]`은 맨 뒤(직전 달)의 값입니다.

```python
history = list(train["air"])
y_pred = []
for month in test.index:
    X_next = pd.DataFrame([[history[-1], history[-2], history[-3], history[-4]]], columns=lags)
    next_value = model.predict(X_next)[0]
    y_pred.append(next_value)
    history.append(next_value)
preds["AR(4)"] = pd.Series(y_pred, index=test.index)
preds["AR(4)"].head(3).round(2)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ 2000년 9월 **64.91**, 10월 **63.12**이면 맞습니다. 10월의 입력 `lag1`에는 9월의 예측값 64.91이 들어갔습니다.

In [ ]:
# ▶ 그대로 실행하세요 — 자기회귀 AR(4) 그림
plot_forecast("AR(4)")

💬 학습 기간의 적합값은 실제 값을 한 달 늦게 따라갑니다. 테스트 기간의 예측은 8월의 높은 값에서 출발해 달마다 조금씩 낮아질 뿐, 계절 모양을 그리지 않습니다. 입력이 직전 4개월뿐이라 1년 전 같은 달의 값이 들어 있지 않기 때문입니다.

### 계절 자기회귀

1년 전 같은 달의 값 `lag12` 하나만 입력으로 씁니다. 처음 12개월은 입력이 없어 뺍니다.

```python
X_train = df.loc[train.index, ["lag12"]].iloc[12:]
y_train = train["air"].iloc[12:]
model = LinearRegression().fit(X_train, y_train)
fits["seasonal AR"] = pd.Series(model.predict(X_train), index=X_train.index)
print(model.intercept_, model.coef_)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ 절편 약 **−0.56**, 계수 약 **1.055**이면 맞습니다.

💬 1년 전 같은 달의 값에 1.055를 곱하고 0.56을 뺍니다. 계수가 1보다 조금 커서, 해마다 늘어나는 만큼을 담습니다.

테스트 기간 12개월의 입력 `lag12`는 1999년 9월 ~ 2000년 8월의 값으로 모두 학습 기간의 실제 값입니다. 그래서 예측값을 이어 쓰지 않고 `predict` 한 번으로 12개월을 냅니다.

```python
X_test = df.loc[test.index, ["lag12"]]
preds["seasonal AR"] = pd.Series(model.predict(X_test), index=test.index)
preds["seasonal AR"].head(3).round(2)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ 2000년 9월 **56.11**이면 맞습니다. 1999년 9월의 값 53.73에 계수를 곱하고 절편을 더한 값입니다.

In [ ]:
# ▶ 그대로 실행하세요 — 계절 자기회귀 그림
plot_forecast("seasonal AR")

💬 seasonal naive 예측과 같은 계절 모양을 조금 키운 값이라, 테스트 기간에는 대체로 실제보다 조금 위에 있습니다.

---
## 5. 추세·계절성 회귀

**추세·계절성 회귀**는 설명 변수가 시점 t와 계절인 회귀 모형입니다. 설명 변수 열을 만듭니다.
- **t:** 시점 번호. 1990년 1월이 1, 2004년 4월이 172입니다. 2001년 9월 이후 기간에도 번호를 이어서 매깁니다.
- **계절성 dummy 변수:** 그 행이 해당 달이면 1, 아니면 0인 열. 기준 달(1월)을 빼고 2~12월에 11개를 만듭니다. 12개를 모두 넣으면 어느 행이든 12개 열의 합이 1이 되어 상수항과 겹치기 때문입니다.

`pd.get_dummies`가 달마다 열을 만들고, `drop_first=True`가 첫 달(1월) 열을 뺍니다. `astype(int)`는 참·거짓을 1·0으로 바꿉니다.

```python
df["t"] = np.arange(1, len(df) + 1)
dummies = pd.get_dummies(df.index.month, prefix="m", drop_first=True).astype(int)
dummies.index = df.index
df = pd.concat([df, dummies], axis=1)
df[["air", "t", "m_2", "m_3", "m_12"]].head(3)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ 1990년 2월 행의 `m_2`만 1이면 맞습니다.

두 모형 모두 학습 기간 128개월 전체로 맞춥니다.

### 선형 추세 모형

시점 번호 t 하나로 직선을 맞춥니다. 식은 여객 = β₀ + β₁ × t입니다. 4주차 회귀와 같고, 설명 변수가 시점 번호일 뿐입니다.

```python
X_train = df.loc[train.index, ["t"]]
y_train = train["air"]
model = LinearRegression().fit(X_train, y_train)
fits["linear"] = pd.Series(model.predict(X_train), index=X_train.index)
print(model.intercept_, model.coef_)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ 절편 약 **35.79**, 기울기 약 **0.176**이면 맞습니다.

💬 한 달에 0.176(십억 여객마일)씩, 1년에 약 2.1씩 늘어나는 직선입니다.

테스트 기간의 t(129~140)를 넣어 예측합니다.

```python
X_test = df.loc[test.index, ["t"]]
preds["linear"] = pd.Series(model.predict(X_test), index=test.index)
preds["linear"].head(3).round(2)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ 2000년 9월 **58.51**이면 맞습니다.

In [ ]:
# ▶ 그대로 실행하세요 — 선형 추세 모형 그림
plot_forecast("linear")

💬 직선은 늘어나는 흐름을 따라가지만, 해마다 되풀이되는 오르내림은 담지 않습니다. 실제 값이 직선 위아래로 크게 오르내립니다.

### 선형 추세 + 계절성 모형

설명 변수는 t와 계절성 dummy 변수 11개입니다. 열 이름이 많으므로 리스트 `cols`로 모아 둡니다. `list(dummies.columns)`는 `m_2` ~ `m_12`입니다.

```python
cols = ["t"] + list(dummies.columns)
X_train = df.loc[train.index, cols]
model = LinearRegression().fit(X_train, y_train)
fits["linear + season"] = pd.Series(model.predict(X_train), index=X_train.index)
print(model.intercept_)
pd.Series(model.coef_, index=cols).round(3)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ 절편 약 **31.35**, `t` **0.171**, `m_2` −2.522, `m_8` **13.34**이면 맞습니다.

💬 **결과 해설** — 4주차처럼 다른 설명 변수가 같을 때의 차이로 읽습니다.
- `t` = 0.171: 같은 달끼리 보면 한 달에 0.171씩 늘어납니다.
- `m_8` = 13.34: 같은 시점의 추세에서 8월은 기준인 1월보다 13.34(십억 여객마일) 많습니다. `m_2` = −2.52: 2월은 1월보다 적습니다.
- 계절성 dummy 변수는 해마다 같은 크기만큼 더하고 뺍니다. 계절 차이가 해마다 같은 크기라고 가정한 것입니다.

```python
X_test = df.loc[test.index, cols]
preds["linear + season"] = pd.Series(model.predict(X_test), index=test.index)
preds["linear + season"].head(3).round(2)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ 2000년 9월 **56.41**이면 맞습니다.

In [ ]:
# ▶ 그대로 실행하세요 — 선형 추세 + 계절성 모형 그림
plot_forecast("linear + season")

💬 적합값이 계절 모양까지 따라가고, 테스트 기간의 예측도 실제 값 가까이에 있습니다.

---
## 6. 테스트 기간의 오차로 고르기

떼어 둔 12개월의 실제 값과 예측값을 비교합니다. **예측 오차**는 실제 값 − 예측값입니다. 양수면 실제보다 적게 예측한 것입니다. seasonal naive 예측의 오차부터 봅니다.

```python
e = y_test - preds["seasonal naive"]
e.head(3).round(2)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ 2000년 9월 오차가 **2.71**이면 맞습니다. 실제 56.44에서 예측 53.73을 뺀 값입니다.

12개 오차를 숫자 하나로 요약합니다.

| 지표 | 계산 | 뜻 |
|---|---|---|
| MSE | 오차 제곱의 평균 | 오차의 크기, 큰 오차에 더 큰 벌점 |
| MAPE | (오차 ÷ 실제 값)의 절댓값 평균 × 100 | 실제 값의 몇 % 틀리는지 |

방법마다 같은 계산을 하므로 함수 `errors`로 만듭니다.

```python
def errors(y_true, y_pred):
    e = y_true - y_pred
    return pd.Series({"MSE": (e ** 2).mean(),
                      "MAPE": (e / y_true).abs().mean() * 100})

errors(y_test, preds["seasonal naive"]).round(2)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ seasonal naive 예측의 MSE **3.62**, MAPE **2.74**이면 맞습니다.

💬 실제 값의 약 2.7% 틀렸습니다.

`preds`에 모은 모든 방법의 오차를 한 표로 모아 MAPE 순서로 놓습니다. `{name: ... for name in preds}`는 방법 이름마다 `errors`를 계산해 모읍니다.

```python
table = pd.DataFrame({name: errors(y_test, preds[name]) for name in preds}).T
table.sort_values("MAPE").round(2)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ 테스트 MAPE가 선형 추세 + 계절성 모형 **1.88**, seasonal naive 예측 2.74, 계절 자기회귀 2.76, 선형 추세 모형 6.87, 자기회귀 AR(4) 12.80, naive 예측 14.22이면 맞습니다.

💬 **결과 해설**
- 테스트 MAPE가 가장 작은 방법을 고릅니다. 선형 추세 + 계절성 모형이 1.88%로 가장 작습니다.
- 1년 전 같은 달의 값이나 계절성 dummy 변수가 없는 방법(naive 예측 14.22%, AR(4) 12.80%, 선형 추세 모형 6.87%)은 크게 틀립니다.

적합값이 있는 방법은 학습 오차와 테스트 오차를 나란히 봅니다. 자기회귀는 입력이 있는 달(AR(4)는 5번째 달부터, 계절 자기회귀는 13번째 달부터)로 계산합니다.

```python
mape = pd.DataFrame({"train": [errors(train["air"], fits[name])["MAPE"] for name in fits],
                     "test": [errors(y_test, preds[name])["MAPE"] for name in fits]},
                    index=list(fits))
mape.round(2)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ 선형 추세 + 계절성 모형 학습 **2.54**·테스트 **1.88**, 자기회귀 AR(4) 학습 **7.50**·테스트 **12.80**이면 맞습니다.

💬 **결과 해설**
- AR(4)는 학습 MAPE가 7.50%인데 테스트 MAPE는 12.80%로 커집니다. 학습 오차는 모형이 이미 본 자료로 계산하므로, 방법은 테스트 오차로 고릅니다.
- 기준선은 테스트 MAPE가 가장 작은 **선형 추세 + 계절성 모형**으로 만듭니다.

---
## 7. 2001년 9월 이후의 기준선

방법을 고른 뒤에는 떼어 두었던 12개월까지 포함한 2001년 8월까지의 140개월 전체로 다시 맞춥니다. 그 식에 2001년 9월 이후 32개월의 t와 계절성 dummy 변수를 넣은 값이 이전 추세가 이어졌을 때의 예측, 곧 **기준선**입니다. t와 달은 2001년 9월 이후에도 미리 알 수 있는 값입니다.

```python
X = df.loc[pre.index, cols]
y = pre["air"]
model = LinearRegression().fit(X, y)
baseline = pd.Series(model.predict(df.loc[post.index, cols]), index=post.index)
pd.DataFrame({"actual": post["air"], "baseline": baseline}).head(3).round(1)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ 2001년 9월의 실제 **39.1**, 기준선 **58.6**이면 맞습니다.

In [ ]:
# ▶ 그대로 실행하세요 — 실제 값과 기준선
plt.figure(figsize=(9, 3.5))
plt.plot(df.loc["1998":].index, df.loc["1998":, "air"], color="gray", label="Actual")
plt.plot(baseline.index, baseline, linestyle="--", label="Baseline")
plt.axvline(pd.Timestamp("2001-09-01"), color="red", linestyle="--")
plt.ylabel("Air RPM (billions)")
plt.legend()
plt.show()

💬 2001년 9월 이후 실제 값(회색)이 기준선(점선)보다 계속 아래에 있습니다. 두 선 사이가 이전 추세에서 벗어난 부분입니다.

---
## 8. 얼마나 줄었고, 언제까지 이어졌나

달마다 실제 값 − 기준선을 `gap`, 기준선 대비 비율(%)을 `gap_pct`로 구합니다.

```python
gap = post["air"] - baseline
gap_pct = gap / baseline * 100
gap_pct.round(1)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ 2001년 9월 **−33.3**, 2004년 4월 **−4.5**이면 맞습니다.

💬 2001년 9월 한 달 동안 항공 여객은 기준선보다 3분의 1이 적었습니다. 차이는 줄어들었지만 2년 반이 지난 2004년 4월에도 4.5% 적습니다.

첫 1년(2001년 9월 ~ 2002년 8월)의 차이를 모두 더하고, 기준선 합계에 대한 비율을 구합니다.

```python
first_year = slice("2001-09", "2002-08")
loss = gap.loc[first_year].sum()
print(loss, loss / baseline.loc[first_year].sum() * 100)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ 약 **−114.9**(십억 여객마일), **−15.6**%이면 맞습니다.

💬 첫 1년 동안 약 1,149억 여객마일, 기준선의 15.6%가 줄었습니다.

**기준선을 무엇으로 잡느냐에 따라 차이의 크기가 달라집니다.** seasonal naive 예측으로 기준선을 잡았다면 2001년 8월까지의 마지막 12개월(2000년 9월 ~ 2001년 8월)을 그대로 되풀이했을 것입니다. 같은 첫 1년의 차이를 구해 봅니다.

```python
snaive_base = pd.Series(pre["air"].iloc[-12:].tolist(), index=post.loc[first_year].index)
loss_snaive = (post.loc[first_year, "air"] - snaive_base).sum()
print(loss_snaive, loss_snaive / snaive_base.sum() * 100)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ 약 **−94.7**, **−13.2**%이면 맞습니다.

💬 **결과 해설**
- seasonal naive 예측의 기준선은 해마다 늘어나던 추세를 빼먹기 때문에, 첫 1년의 손실을 947억 여객마일(13.2%)로 약 18% 작게 잡습니다.
- 손실 추정은 지원·감편 규모의 근거가 되므로, 기준선을 만들 방법은 테스트 오차로 따져서 고릅니다.

### 결정

> **2001년 9월 이후 미국 항공 여객은 첫 1년 동안 기준선보다 약 1,149억 여객마일(15.6%) 적었고, 2년 반 뒤에도 약 5% 적었습니다. 차이는 일시적이 아니라 계속됐습니다.**

- 기준선은 2001년 8월까지의 마지막 12개월 테스트 기간에서 오차가 가장 작은 선형 추세 + 계절성 모형(MAPE 1.88%)으로 만들었습니다.
- 기준선도 예측이므로 테스트 기간 정도(2% 안팎)의 오차는 있습니다. 2001년 9월의 33% 감소는 그보다 훨씬 큽니다.
- 2001년 9월 이후 32개월은 테스트한 예측 기간(12개월)보다 깁니다. 예측 기간이 길어질수록 오차가 커지므로 2년차부터의 기준선은 그만큼 덜 정확할 수 있고, 그 사이에 경기 둔화 같은 다른 변화가 있었다면 그 몫도 차이에 섞입니다.

---
## 9. 실습 과제

### 📝 과제 1 · 계절 자기회귀로 기준선을 만들면

6절에서 고르지 않은 계절 자기회귀로 첫 1년(2001년 9월 ~ 2002년 8월)의 기준선을 만들면 첫 1년 손실은 얼마로 나오나요? 선형 추세 + 계절성 모형의 결과와 비교하세요.

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: 4절의 계절 자기회귀 셀처럼 lag12 하나로 pre 전체(처음 12개월 제외)를 학습하고, 첫 1년의 lag12로 예측한 뒤 8절의 첫 1년 셀처럼 차이의 합과 비율을 구합니다.


✏️ **나의 답:** (여기를 두 번 눌러 적습니다)

### 📝 과제 2 · 2년차의 손실

2년차(2002년 9월 ~ 2003년 8월)의 차이 합계와 기준선 대비 비율을 구하세요. 첫 1년과 비교해 차이가 얼마나 줄었나요?

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: 8절의 first_year처럼 2002-09부터 2003-08까지의 slice를 만들고, 같은 계산을 합니다.


✏️ **나의 답:**

---
## ✅ 정리

| 배운 것 | 핵심 |
|---|---|
| 자료의 열 | `Air RPM (000s)` 항공 유상 여객마일(천), `Rail PM` 철도 여객마일, `VMT (billions)` 자동차 주행거리 |
| 시계열 만들기 | 날짜 글자 → `pd.to_datetime` → index. 172개월, 늘어나는 추세와 12개월 계절성 |
| 기간 나누기 | 기준선은 2001년 8월까지의 140개월로만, 그 마지막 12개월이 테스트 기간 |
| naive·seasonal naive 예측 | 마지막 값 / 마지막 12개월을 그대로 씀 |
| 자기회귀 | `shift`로 lag 열. AR(4)는 직전 4개월을 입력으로 다단계 예측, 계절 자기회귀는 `lag12` 하나 |
| 추세·계절성 회귀 | t, 계절성 dummy 변수 11개(1월 기준). 2001년 9월 이후 행의 열도 미리 알 수 있음 |
| 오차 지표 | MSE는 오차 제곱의 평균, MAPE는 실제 값의 몇 % 틀리는지 |
| 비교 | 테스트 MAPE: 선형 추세 + 계절성 모형 1.88 · seasonal naive 예측 2.74 · 계절 자기회귀 2.76 · 선형 추세 모형 6.87 · 자기회귀 AR(4) 12.80 · naive 예측 14.22 |
| 학습 오차와 테스트 오차 | AR(4)는 학습 7.50% → 테스트 12.80%, 방법은 테스트 오차로 고름 |
| 기준선과의 차이 | 2001-09 −33.3%, 첫 1년 −1,149억 여객마일(−15.6%), 2004-04에도 −4.5% |
| 기준선의 무게 | seasonal naive 예측의 기준선이면 첫 1년 −13.2%로 약 18% 작게 잡힘 |

**랩:** 같은 자료의 철도 여객마일로 같은 절차를 밟습니다. 철도 여객도 기준선에서 벗어났을까요?